# True base rate of "next day lower in calories" — first pair per user

**Purpose.** The pre-registration assumes the random base rate of "Day 2 lower than Day 1" is **50%**. That holds only if, in the real data, the later day of a consecutive pair is genuinely lower in calories about half the time.

This notebook measures that base rate directly from the calorie totals, over the **exact sample the
study uses**: the **first consecutive pair per user** — one pair each. That is the sample the metric is computed over,
so its base rate is the relevant null for the design.

It reports the count, the rate, a 95% confidence interval, an exact binomial test against the 0.5 null.


In [ ]:
# --- Config + parsing helpers (same logic as the main pipeline) ---
import pandas as pd
import numpy as np
import json

# Path to the MyFitnessPal diaries TSV (user_id \t date \t meals_json \t totals_json)
TSV_PATH = "mfp-diaries.tsv"

# How many users to scan. None = ALL users (best base-rate estimate). Set an int for a quick test run.
MAX_USERS = None
CHUNKSIZE = 50_000

def parse_json_safe(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    try:
        return json.loads(s)
    except Exception:
        return None

def parse_number(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip().replace(",", "")
    if s == "":
        return None
    try:
        return float(s)
    except Exception:
        return None

def extract_total_calories(totals_obj):
    # totals_obj["total"] is a list of {name, value}
    if not isinstance(totals_obj, dict):
        return None
    total_list = totals_obj.get("total")
    if not isinstance(total_list, list):
        return None
    for item in total_list:
        if isinstance(item, dict) and item.get("name") == "Calories":
            return parse_number(item.get("value"))
    return None


In [2]:
# --- Load all usable days per user (date + calories only; food text not needed here) ---
# user_days: uid -> list of (date_dt, calories_total)
user_days = {}

reader = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=["user_id", "date", "meals_json", "totals_json"],
    chunksize=CHUNKSIZE,
    dtype={"user_id": "Int64", "date": "string"},
    engine="c",
)

stop = False
for chunk_idx, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=["user_id", "date"])
    chunk["user_id"] = chunk["user_id"].astype(int)

    chunk["totals"] = chunk["totals_json"].apply(parse_json_safe)
    chunk["calories_total"] = chunk["totals"].apply(extract_total_calories)
    chunk["date_dt"] = pd.to_datetime(chunk["date"], errors="coerce")

    # keep rows with a real date and a parseable calorie total
    chunk = chunk.dropna(subset=["date_dt", "calories_total"])

    for uid, dt, cal in zip(chunk["user_id"], chunk["date_dt"], chunk["calories_total"]):
        uid = int(uid)
        if MAX_USERS is not None and uid not in user_days and len(user_days) >= MAX_USERS:
            continue
        user_days.setdefault(uid, []).append((dt, float(cal)))

    n_users = len(user_days)
    print(f"Chunk {chunk_idx}: users_seen={n_users}, rows_in_chunk={len(chunk)}")
    if MAX_USERS is not None and n_users >= MAX_USERS:
        stop = True
    if stop:
        break

print("Total users loaded:", len(user_days))


Chunk 0: users_seen=858, rows_in_chunk=49994
Chunk 1: users_seen=1726, rows_in_chunk=49998
Chunk 2: users_seen=2568, rows_in_chunk=50000
Chunk 3: users_seen=3363, rows_in_chunk=49996
Chunk 4: users_seen=4186, rows_in_chunk=49998
Chunk 5: users_seen=5047, rows_in_chunk=49999
Chunk 6: users_seen=5880, rows_in_chunk=49998
Chunk 7: users_seen=6737, rows_in_chunk=49995
Chunk 8: users_seen=7593, rows_in_chunk=49997
Chunk 9: users_seen=8414, rows_in_chunk=50000
Chunk 10: users_seen=9240, rows_in_chunk=49999
Chunk 11: users_seen=9895, rows_in_chunk=37187
Total users loaded: 9895


In [3]:
# --- Build the FIRST consecutive pair per user and flag whether the LATER day is lower in calories ---
# A "consecutive pair" is two days exactly 1 calendar day apart.
# later_lower = (later day's calories) < (earlier day's calories)   [strict, matches "lower"]
# tie         = equal calories on both days
# One pair per user: the first consecutive pair, matching the study design.

first_pairs = []

for uid, rows in user_days.items():
    dfu = (pd.DataFrame(rows, columns=["date_dt", "calories_total"])
             .drop_duplicates(subset=["date_dt"])
             .sort_values("date_dt")
             .reset_index(drop=True))
    for i in range(len(dfu) - 1):
        if (dfu.loc[i + 1, "date_dt"] - dfu.loc[i, "date_dt"]).days == 1:
            earlier = dfu.loc[i, "calories_total"]
            later   = dfu.loc[i + 1, "calories_total"]
            first_pairs.append({
                "user_id": uid,
                "earlier_date": dfu.loc[i, "date_dt"],
                "earlier_cal": earlier,
                "later_cal": later,
                "delta_later_minus_earlier": later - earlier,
                "later_lower": bool(later < earlier),
                "tie": bool(later == earlier),
            })
            break  # first consecutive pair only

first_df = pd.DataFrame(first_pairs)
print("First-pair-per-user:", len(first_df))
first_df.head()


First-pair-per-user: 9161


,user_id,earlier_date,earlier_cal,later_cal,delta_later_minus_earlier,later_lower,tie
0,1,2014-09-14,2924.0,2430.0,-494.0,True,False
1,2,2015-01-12,2117.0,1327.0,-790.0,True,False
2,3,2014-09-14,2270.0,1628.0,-642.0,True,False
3,4,2014-10-21,1069.0,984.0,-85.0,True,False
4,5,2014-09-14,3405.0,4127.0,722.0,False,False


In [4]:
# --- Base-rate summary with 95% CI and exact binomial test vs the 0.5 null ---
from scipy.stats import binomtest

def base_rate_report(df, name, threshold_margin=0.03):
    n = len(df)
    if n == 0:
        print(f"[{name}] no pairs found.")
        return None
    n_lower = int(df["later_lower"].sum())          # strict: later < earlier
    n_ties  = int(df["tie"].sum())
    rate = n_lower / n

    bt = binomtest(n_lower, n, 0.5, alternative="two-sided")
    ci_low, ci_high = bt.proportion_ci(confidence_level=0.95)

    # If ties were instead counted as "lower" (non-strict), how much would the rate move?
    rate_with_ties = (n_lower + n_ties) / n

    # What the "baseline + margin" threshold becomes when anchored to the measured rate
    measured_threshold = rate + threshold_margin

    print(f"  pairs (n)                  : {n}")
    print(f"  later day lower (strict)   : {n_lower}")
    print(f"  exact ties (equal cals)    : {n_ties}")
    print(f"  BASE RATE  P(later<earlier): {rate:.4f}   (95% CI {ci_low:.4f}-{ci_high:.4f})")
    print(f"  --> 50% assumption {'HOLDS' if (ci_low <= 0.5 <= ci_high) else 'is REJECTED'} at 95% (CI {'includes' if (ci_low <= 0.5 <= ci_high) else 'excludes'} 0.50)")
    print(f"  --> pre-reg threshold uses 0.50+{threshold_margin:.0%} = {0.5+threshold_margin:.2%}")
    return {
        "name": name, "n": n, "n_lower": n_lower, "n_ties": n_ties,
        "base_rate": rate, "ci_low": ci_low, "ci_high": ci_high,
        "p_vs_0.5": bt.pvalue, "rate_with_ties": rate_with_ties,
        "measured_threshold": measured_threshold,
    }

res_first = base_rate_report(first_df, "FIRST pair per user")
summary_df = pd.DataFrame([r for r in (res_first,) if r is not None])



  pairs (n)                  : 9161
  later day lower (strict)   : 4613
  exact ties (equal cals)    : 66
  BASE RATE  P(later<earlier): 0.5035   (95% CI 0.4933-0.5138)
  --> 50% assumption HOLDS at 95% (CI includes 0.50)
  --> pre-reg threshold uses 0.50+3% = 53.00%


## Base rate over the study sample — first 100 pairs

The study uses only **100** users (one first-consecutive-pair each). Above we measured the
base rate over *all* users as a population estimate; here we report the same metric restricted
to the **first 100 pairs** — the actual sample the diet experiment is run on. This is the
relevant null for the design and matches `diet_ground_truth.csv`.


In [5]:
# --- Same base-rate report, restricted to the first 100 pairs (the study sample) ---
first_100_df = first_df.head(100)

res_first_100 = base_rate_report(first_100_df, "FIRST 100 pairs (study sample)")

# Day-2-lower headline figure for the 100-pair sample
print(f"\n  Ground-truth % Day 2 (later day) lower over first 100 pairs: "
      f"{100 * first_100_df['later_lower'].mean():.1f}%")


  pairs (n)                  : 100
  later day lower (strict)   : 49
  exact ties (equal cals)    : 2
  BASE RATE  P(later<earlier): 0.4900   (95% CI 0.3886-0.5920)
  --> 50% assumption HOLDS at 95% (CI includes 0.50)
  --> pre-reg threshold uses 0.50+3% = 53.00%

  Ground-truth % Day 2 (later day) lower over first 100 pairs: 49.0%
